In [2]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score
import pandas as pd 
import shap 
import xgboost 
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt 

/Users/sorenbasnet/Documents/Github/STAT_443_FINAL_PROJECT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = pd.read_csv("/Users/sorenbasnet/Documents/Github/STAT_443_FINAL_PROJECT/data/prediction_death_5_years/data_patient_file_cna_rai.txt", sep=r'\s+',index_col=0)
data.head()

,ONCOTREE_CODE,DISEASE_TYPE,NORMAL_MEAN_COVERAGE,TUMOR_MEAN_COVERAGE,TUMOR_SAMPLE_PURITY,TUMOR_SAMPLE_PLOIDY,CANCER_TYPE,CANCER_TYPE_DETAILED,SOMATIC_STATUS,CLL_EPITYPE,TUMOR_MOLECULAR_SUBTYPE,SEQUENCING_PLATFORM,RNA_SEQUENCING,RRBS,ARRAY450K,M.CLL_WES,U.CLL_WES,EC_DISCOVERY,EC_EXTENSION,IGLV3_21_R110,U1_STATUS,TMB_NONSYNONYMOUS,AGE,AGE_SAMPLING,SEX,OS_STATUS,OS_MONTHS,DEATH_DAYS,COHORT,IGHV_MUTATION_STATUS,IGHV_IDENTITY_PERCENTAGE,TREATMENT_STATUS,PRIOR_TREATMENT_CATEGORY,TREATMENT_AFTER_SAMPLING,EXPRESSION_CLUSTER,FFS_STATUS,FFS_MONTHS,11q_status,17p_status,17q_status,18p_status,19q_status,20p_status,2p_status,4p_status,6q_status,8p_status,8q_status,tri12_status,TREATMENT_STATUS_AT_SAMPLING,RAI_AT_SAMPLING,FIRST_TREATMENT_AFTER_SAMPLING,surv_5yrs,death_5yr
SAMPLE_ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
CRC_0001,CLLSLL,CLL,63.0,79.7,0.92,2.00,Mature__B_Cell__Neoplasms,Chronic__Lymphocytic__Leukemia_Small__Lymphocy...,Matched,n_CLL,U_CLL,WES,Yes,Yes,No,No,Yes,Yes,No,No,WT,0.866667,44.0,46.0,Female,DECEASED,147.19,4477.0,UCSD,unmutated,100.00,Pre_treatment,Untreated,Chemo__+__Ab,EC_u1,1:Failure,43.17,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Gain,Unchanged,Untreated,0.0,Chemo__+__Ab,60.00,0.0
CRC_0002,CLLSLL,CLL,70.1,135.5,0.94,2.03,Mature__B_Cell__Neoplasms,Chronic__Lymphocytic__Leukemia_Small__Lymphocy...,Matched,n_CLL,U_CLL,WES,Yes,Yes,No,No,Yes,Yes,No,No,WT,0.766667,55.0,56.0,Male,DECEASED,154.49,4699.0,UCSD,unmutated,100.00,Pre_treatment,Untreated,Chemo__+__Ab,EC_u1,1:Failure,66.64,Unchanged,Unchanged,Unchanged,Loss,Unchanged,Unchanged,Gain,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Untreated,0.0,Chemo__+__Ab,60.00,0.0
CRC_0003,CLLSLL,CLL,77.2,141.1,0.83,2.05,Mature__B_Cell__Neoplasms,Chronic__Lymphocytic__Leukemia_Small__Lymphocy...,Matched,n_CLL,U_CLL,WES,Yes,Yes,No,No,Yes,Yes,No,No,WT,0.233333,63.0,63.0,Female,DECEASED,51.25,1559.0,UCSD,unmutated,100.00,Pre_treatment,Untreated,Chemo__+__Ab,EC_u2,1:Failure,27.75,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Gain,Untreated,0.0,Chemo__+__Ab,51.25,1.0
CRC_0004,CLLSLL,CLL,60.2,146.0,0.92,2.00,Mature__B_Cell__Neoplasms,Chronic__Lymphocytic__Leukemia_Small__Lymphocy...,Matched,m_CLL,M_CLL,WES,Yes,Yes,No,Yes,No,Yes,No,No,WT,1.000000,51.0,51.0,Male,LIVING,183.55,0.0,UCSD,mutated,92.28,Pre_treatment,Untreated,Chemo__+__Ab,EC_m4,1:Failure,92.48,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Untreated,0.0,Chemo__+__Ab,60.00,0.0
CRC_0005,CLLSLL,CLL,106.5,82.1,0.87,2.05,Mature__B_Cell__Neoplasms,Chronic__Lymphocytic__Leukemia_Small__Lymphocy...,Matched,n_CLL,U_CLL,WES,Yes,Yes,No,No,Yes,Yes,No,No,WT,0.766667,36.0,37.0,Male,LIVING,164.35,0.0,UCSD,unmutated,100.00,Pre_treatment,Untreated,Chemo__+__Ab,EC_u2,1:Failure,59.57,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Unchanged,Gain,Untreated,0.0,Chemo__+__Ab,60.00,0.0


In [4]:
data.columns

Index(['ONCOTREE_CODE', 'DISEASE_TYPE', 'NORMAL_MEAN_COVERAGE',
       'TUMOR_MEAN_COVERAGE', 'TUMOR_SAMPLE_PURITY', 'TUMOR_SAMPLE_PLOIDY',
       'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'SOMATIC_STATUS', 'CLL_EPITYPE',
       'TUMOR_MOLECULAR_SUBTYPE', 'SEQUENCING_PLATFORM', 'RNA_SEQUENCING',
       'RRBS', 'ARRAY450K', 'M.CLL_WES', 'U.CLL_WES', 'EC_DISCOVERY',
       'EC_EXTENSION', 'IGLV3_21_R110', 'U1_STATUS', 'TMB_NONSYNONYMOUS',
       'AGE', 'AGE_SAMPLING', 'SEX', 'OS_STATUS', 'OS_MONTHS', 'DEATH_DAYS',
       'COHORT', 'IGHV_MUTATION_STATUS', 'IGHV_IDENTITY_PERCENTAGE',
       'TREATMENT_STATUS', 'PRIOR_TREATMENT_CATEGORY',
       'TREATMENT_AFTER_SAMPLING', 'EXPRESSION_CLUSTER', 'FFS_STATUS',
       'FFS_MONTHS', '11q_status', '17p_status', '17q_status', '18p_status',
       '19q_status', '20p_status', '2p_status', '4p_status', '6q_status',
       '8p_status', '8q_status', 'tri12_status',
       'TREATMENT_STATUS_AT_SAMPLING', 'RAI_AT_SAMPLING',
       'FIRST_TREATMENT_AFTER_SA

In [6]:
data = data.dropna(subset=["death_5yr"])

In [7]:
X = data.drop(columns=['ONCOTREE_CODE', 'DISEASE_TYPE', 'NORMAL_MEAN_COVERAGE',
       'TUMOR_MEAN_COVERAGE', 'TUMOR_SAMPLE_PURITY', 'TUMOR_SAMPLE_PLOIDY',
       'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'SOMATIC_STATUS', 'CLL_EPITYPE',
       'TUMOR_MOLECULAR_SUBTYPE', 'SEQUENCING_PLATFORM', 'RNA_SEQUENCING',
       'RRBS', 'ARRAY450K', 'M.CLL_WES', 'U.CLL_WES', 'EC_DISCOVERY',
       'EC_EXTENSION', 'IGLV3_21_R110', 'U1_STATUS', 'TMB_NONSYNONYMOUS',
       'AGE', 'AGE_SAMPLING', 'SEX', 'OS_STATUS', 'OS_MONTHS', 'DEATH_DAYS',
       'COHORT', 'IGHV_MUTATION_STATUS', 'IGHV_IDENTITY_PERCENTAGE',
       'TREATMENT_STATUS', 'PRIOR_TREATMENT_CATEGORY',
       'TREATMENT_AFTER_SAMPLING', 'EXPRESSION_CLUSTER', 'FFS_STATUS',
       'FFS_MONTHS', 
       'TREATMENT_STATUS_AT_SAMPLING', 'RAI_AT_SAMPLING',
       'FIRST_TREATMENT_AFTER_SAMPLING', 'surv_5yrs', 'death_5yr'])
y = data["death_5yr"]

In [8]:
# This automatically generated dummies for categorical data 
X = pd.get_dummies(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y)

In [10]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

,"solver solver: {'svd', 'lsqr', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'lsqr': Least squares solution. Can be combined with shrinkage or custom covariance estimator. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator... versionchanged:: 1.2 `solver=""svd""` now has experimental Array API support. See the :ref:`Array API User Guide ` for more details.",'svd'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter.This should be left to None if `covariance_estimator` is used.Note that shrinkage works only with 'lsqr' and 'eigen' solvers.For a usage example, see:ref:`sphx_glr_auto_examples_classification_plot_lda.py`.",None
,"priors priors: array-like of shape (n_classes,), default=NoneThe class prior probabilities. By default, the class proportions areinferred from the training data.",None
,"n_components n_components: int, default=NoneNumber of components (<= min(n_classes - 1, n_features)) fordimensionality reduction. If None, will be set tomin(n_classes - 1, n_features). This parameter only affects the`transform` method.For a usage example, see:ref:`sphx_glr_auto_examples_decomposition_plot_pca_vs_lda.py`.",None
,"store_covariance store_covariance: bool, default=FalseIf True, explicitly compute the weighted within-class covariancematrix when solver is 'svd'. The matrix is always computedand stored for the other solvers... versionadded:: 0.17",False
,"tol tol: float, default=1.0e-4Absolute threshold for a singular value of X to be consideredsignificant, used to estimate the rank of X. Dimensions whosesingular values are non-significant are discarded. Only used ifsolver is 'svd'... versionadded:: 0.17",0.0001
,"covariance_estimator covariance_estimator: covariance estimator, default=NoneIf not None, `covariance_estimator` is used to estimatethe covariance matrices instead of relying on the empiricalcovariance estimator (with potential shrinkage).The object should have a fit method and a ``covariance_`` attributelike the estimators in :mod:`sklearn.covariance`.if None the shrinkage parameter drives the estimate.This should be left to None if `shrinkage` is used.Note that `covariance_estimator` works only with 'lsqr' and 'eigen'solvers... versionadded:: 0.24",None


In [11]:
probs = lda.predict_proba(X_test)[:,1]

In [12]:
auc = roc_auc_score(y_test, probs)

In [13]:
print("AUC : ", auc)

AUC :  0.6322562358276644


# The AUC curve score if 0.63. should look into what GUIDE does. 